# Run a simulation

**Model:** `pfc_models.models.stdpfc_cpu`  
**Written against:** pfc-models 0.0.0  
**Takeaway:** how to build a CPU geometry, set standard-PFC model parameters, step the solver and view the density field

The state point below is the image, under the α = 1 parameter map of
`papers/2026-log-pfc/derivations/c1-positive-density/01-model-parameter-conversion.md`
(§5.2), of the state point used in `examples/logpfc_cpu/run-a-simulation.ipynb`, so the two
notebooks sit at corresponding points of the two branches.

> **The field is `n`, not `phi`.** `n = 0` is the reference liquid and `n` is not
> bounded below; there is no positivity constraint and no log clamp in this model.
> The cubic coefficient `g` is retained — see `model.md` for the paired run that
> verifies it is removable by the shift `n → n + g/3`.

> **Seeding.** The hot initial-condition generators draw from `np.random.default_rng(seed)`
> and force a positive mean, which suits the log family; the perfect-crystal generators used
> here are additive about `meanDensity` and accept the negative mean this model needs.


In [ ]:
# numerics
import numpy as np

# plotting
import plotly.io as pio

# Set the global default renderer
pio.renderers.default = "png"

from importlib import reload

In [ ]:
# density plotting
import pfc_models.viz.heatmap as _densityHeatmap
reload(_densityHeatmap)
from pfc_models.viz.heatmap import *

In [ ]:
# Initial conditions
import pfc_models.fields.initial_conditions_2d as _sim_initial_conditions
reload(_sim_initial_conditions)
from pfc_models.fields.initial_conditions_2d import *

In [ ]:
# Simulation
import pfc_models.models.stdpfc_cpu.simulation as Simulation
reload(Simulation)
from pfc_models.models.stdpfc_cpu.simulation import *

# Simulation Builder

In [ ]:
# geometry for verifiable derivatives
Nx = 66*4
Ny = 66*4
q0_star = 1.0
a = 4*np.pi/3**0.5 / q0_star
h = 4*np.pi / q0_star
Mx = 7*4
My = 4*4
Lx = Mx * a
Ly = My * h

# standard-PFC state point, mapped from the log-PFC example at alpha = 1
# (derivation 01-model-parameter-conversion.md, section 5.2):
#   beta_s = (3/4)^(1/2) * beta_l
#   eps_s  = (3/4)^(1/2) * (eps_l + 2*g_l + 4)
#   g_s    = (3/4)^(3/4) * (g_l + 5/2)
#   n      = (3/4)^(1/4) * (phi - 1)
c12, c34, c14 = (3/4)**0.5, (3/4)**0.75, (3/4)**0.25
beta_l, epsilon_l, g_l = 0.5, -0.9, -2.0      # the log-PFC example's parameters

# model parameters
Gamma_val = 1.0
beta_val = c12 * beta_l                       #  0.433013
q0 = q0_star
epsilon_val = c12 * (epsilon_l + 2*g_l + 4)   # -0.779423
g_val = c34 * (g_l + 2.5)                     #  0.402964
meanDensity_S = c14 * (0.55 - 1)              # -0.418772
meanDensity_V = c14 * (0.30 - 1)              # -0.651423
relStd = 0.1
kMax = np.sqrt(3) / 2
dt = 1e-1

geometry = Build2DGeometry(Nx=Nx, Ny=Ny, Lx=Lx, Ly=Ly)
modelParameters = BuildStdPFCModelParameters(
  Gamma=Gamma_val,
  beta=beta_val,
  q0=q0,
  epsilon=epsilon_val,
  g=g_val,
  dt=dt,
  meanDensity=meanDensity_S
)

sim = BuildNumpyCPU_Simulation(modelParameters, geometry, nThreads=12)
nInit = GeneratePerfectCrystalInitialConditions(geometry, meanDensity_S, A0=c14*0.19, q0=q0)
# nInit = GeneratePerfectCrystalInitialWithCentralVacancy(geometry, meanDensity_S, meanDensity_V, A0=c14*0.15, q0=q0)
sim.State.n[...] = nInit
print(
    geometry.X.shape,
    geometry.KX.shape,
    geometry.X[0,1], geometry.Y[1,0],
  )

X, Y = sim.Geometry.X, sim.Geometry.Y
GetDensityHeatmap(sim.n, X, Y, title='n', showScale=True).show(renderer="notebook")

# Test Simulation

In [ ]:
totalT = 1000.0
steps = 100
miniSteps = int(totalT / steps / sim.Model.dt)
print(f'Running {steps} steps of {miniSteps} mini-steps each (totalT = {totalT})')
for i in range(steps):
  for _ in range(miniSteps):
    sim.TimeStep()
  f = sim.CalculateFreeEnergy()
  print(f'Step {i}: f = {f:0.15f}, n.min = {sim.n.min()}, n.max = {sim.n.max()}, n.mean = {sim.n.mean()}')

GetDensityHeatmap(sim.State.n_next, X, Y, title='n(t + ΔT)', showScale=True).show()